# Prepare the Data for Machine Learning Algorithms



In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [21]:
import os
from sklearn.datasets import fetch_openml

DATA_PATH = '../data/raw/credit_g.csv'

if not os.path.exists(DATA_PATH):
    raw = fetch_openml('credit-g', version=1, as_frame=True)
    os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)
    raw.frame.to_csv(DATA_PATH, index=False)

df = pd.read_csv(DATA_PATH, dtype={'class': 'category'})
df = df.rename(columns={'class': 'target'})


In [22]:
from sklearn.model_selection import train_test_split

# Since classes are 70/30, stratify preserve the ratio
X_train, X_test, y_train, y_test = train_test_split(df.drop('target', axis=1), df['target'], stratify=df['target'], random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(750, 20)
(250, 20)
(750,)
(250,)


Create a pipeline to perform transformations such as log ones, encode categorical variables, and 

In [30]:
ord_cols = {
    'checking_status': ['<0', '0<=X<200', '>=200', 'no checking'],
    'savings_status': ['<100', '100<=X<500', '500<=X<1000', '>=1000', 'no known savings'],
    'employment': ['unemployed', '<1', '1<=X<4', '4<=X<7', '>=7']
}
nom_cols = ['credit_history', 'purpose', 'housing', 'job', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'own_telephone', 'foreign_worker']

In [31]:
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder(categories=[ord_cols[c] for c in ord_cols])
X_train[list(ord_cols.keys())] = enc.fit_transform(X_train[list(ord_cols.keys())])
X_test[list(ord_cols.keys())] = enc.fit_transform(X_test[list(ord_cols.keys())])

In [32]:
X_train = pd.get_dummies(X_train, columns=nom_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=nom_cols, drop_first=True)

In [33]:
X_train['credit_amount'] =  X_train['credit_amount'].apply(lambda x: np.log(x + 1))
X_test['credit_amount'] =  X_test['credit_amount'].apply(lambda x: np.log(x + 1))



In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

Or this

In [24]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.compose import ColumnTransformer


num_preprocessor = Pipeline(steps=[
    ('log', FunctionTransformer(np.log1p, validate=True)),  # Log transformation
    ('scaler', StandardScaler())  # Standard scaling
])

ord_preprocessor = Pipeline(steps=[
    ('encoder', OrdinalEncoder(categories=[ord_cols[c] for c in ord_cols]))  # Ordinal encoding
])

nom_preprocessor = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encoding
])

In [25]:
preprocessor = ColumnTransformer(transformers=[
    ('num', num_preprocessor, num_cols),
    ('ord', ord_preprocessor, list(ord_cols.keys())),
    ('nom', nom_preprocessor, nom_cols)
])

In [26]:
## SAVEEE

In [ ]:
## DELETE, JUST FOR TEST 
from sklearn.linear_model import LogisticRegression 

model = LogisticRegression(max_iter=1000, random_state=42)

model.fit(X_train, y_train)  # Fit on training data
y_pred = model.predict(X_test)  # Predict on test data

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         bad       0.62      0.48      0.54        75
        good       0.80      0.87      0.83       175

    accuracy                           0.76       250
   macro avg       0.71      0.68      0.69       250
weighted avg       0.74      0.76      0.75       250



ValueError: could not convert string to float: '>=200'